# 00 — Quickstart

展示 `simulation_smile` 的三种组装方式。

> 当前为架构骨架：`core/` 接口可用，各成像环节为 `NotImplementedError` 占位。
> 本 notebook 的调用形态已定型，填完各环节实现后即可跑通。

In [ ]:
import numpy as np
import simulation_smile as sm
from simulation_smile import SimulationPipeline, configs as C
from simulation_smile.optics import PupilPSF
from simulation_smile.illumination import UniformField
from simulation_smile.fluorophore import BlinkingModel
from simulation_smile.camera import SCMOSCamera
from simulation_smile.noise import Poisson, ReadNoise, Background
from simulation_smile import viz

## 方式 1：预设场景一键起步

In [ ]:
from simulation_smile import presets

pipe = presets.smlm_storm()
movie = pipe.render_movie(n_frames=50)   # -> (50, H, W)
# viz.animate_movie(movie)

## 方式 2：手动拼装，替换任意环节

In [ ]:
grid = C.Grid(pixel_size=108e-9, size=64, oversampling=4)

pipe = (SimulationPipeline(grid=grid)
        .with_optics(PupilPSF(C.PSFConfig(na=1.49, wavelength=668e-9, zernike={5: 0.3}), grid=grid))
        .with_illumination(UniformField(C.IlluminationConfig()))
        .with_photophysics(BlinkingModel(C.FluorophoreConfig(k_on=0.5, k_off=20, k_bleach=0.01)))
        .with_camera(SCMOSCamera(C.CameraConfig(qe=0.95)))
        .with_noise([Background(C.NoiseConfig(background=5.0)), Poisson(), ReadNoise(C.NoiseConfig(read_sigma=1.5))]))

# frame = pipe.render_frame(0)
# viz.show_frame(frame)

## 方式 3：单独调试某一环节（不进管线）

In [ ]:
psf_model = PupilPSF(C.PSFConfig(na=1.49, wavelength=668e-9), grid=grid)
z = np.linspace(-1e-6, 1e-6, 21)
# stack = psf_model.psf_stack(z)   # -> (21, H, W)
# viz.show_psf(stack[10])